<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습1: Scaled Dot-Product Attention 직접 구현

이 예제는 **Scaled Dot-Product Attention이 어떤 토큰을 얼마나 참고하여 새로운 표현을 만드는지** 단계별로 확인하는 실습입니다.

문장 「철수 영희 책 주었다」의 각 토큰에 간단한 **Q(Query), K(Key), V(Value)** 벡터가 있다고 가정합니다.

특히 **「주었다」가 다른 토큰을 얼마나 참고하는지** 계산 과정을 직접 확인합니다.

### 계산 과정

**① 관련도 계산 → ② Scaling → ③ Softmax → ④ Value 가중합**


쉽게 말하면,

> **어떤 단어와 관련 있는지 계산하고 → 관련도를 가중치로 바꾸고 → 관련 있는 단어의 정보를 많이 가져와 섞습니다.**

### 실행하면서 확인할 것

- **Attention Score**: 어떤 토큰과 얼마나 관련 있는가?
- **Attention Weight**: 각 토큰을 얼마나 참고할 것인가?
- **Output**: 참고한 정보들을 합쳐 어떤 새로운 표현이 만들어지는가?
- **Softmax 결과의 합**이 1인지 확인

> **참고:** 이 예제의 Q, K, V 값은 Attention의 계산 원리를 쉽게 이해하기 위해 임의로 만든 교육용 값입니다. 실제 Transformer에서는 학습된 가중치 \(W_Q, W_K, W_V\)를 이용하여 Q, K, V를 생성합니다.

In [ ]:
import numpy as np


# ============================================================
# 0. Softmax 함수
# ============================================================
def softmax(x, axis=-1):
    """
    점수(score)를 0~1 사이의 가중치로 변환합니다.
    각 행의 가중치 합은 1이 됩니다.
    """

    # 수치적으로 안정적인 계산을 위해 각 행의 최댓값을 뺍니다.
    x = x - np.max(x, axis=axis, keepdims=True)

    exp_x = np.exp(x)

    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


# ============================================================
# 1. Scaled Dot-Product Attention
# ============================================================
def scaled_dot_product_attention(Q, K, V, mask=None):

    # Query와 Key 벡터의 차원
    d_k = Q.shape[-1]

    # --------------------------------------------------------
    # ① Query와 Key의 관련도 계산
    #
    # Q : (토큰 수, d_k)
    # K.T : (d_k, 토큰 수)
    #
    # 결과 scores : (토큰 수, 토큰 수)
    # --------------------------------------------------------
    raw_scores = Q @ K.T

    # --------------------------------------------------------
    # ② Scaling
    #
    # score가 너무 커지는 것을 막기 위해 sqrt(d_k)로 나눕니다.
    # --------------------------------------------------------
    scores = raw_scores / np.sqrt(d_k)

    # --------------------------------------------------------
    # ③ Mask
    #
    # False인 위치는 거의 선택되지 않도록
    # 매우 작은 값(-1e9)을 넣습니다.
    # --------------------------------------------------------
    if mask is not None:
        scores = np.where(mask, scores, -1e9)

    # --------------------------------------------------------
    # ④ Softmax
    #
    # 관련도 점수를 Attention 가중치로 변환합니다.
    # 각 행의 합은 1이 됩니다.
    # --------------------------------------------------------
    weights = softmax(scores, axis=-1)

    # --------------------------------------------------------
    # ⑤ Value의 가중합
    #
    # 중요한 Value는 많이 반영하고
    # 덜 중요한 Value는 적게 반영합니다.
    # --------------------------------------------------------
    output = weights @ V

    # 실습에서는 중간 계산값도 함께 반환
    return raw_scores, scores, weights, output


# ============================================================
# 2. 아주 작은 예제 만들기
# ============================================================

tokens = ["철수", "영희", "책", "주었다"]

# 이해를 쉽게 하기 위해 각 토큰을
# 2차원 Q, K, V 벡터로 단순화했습니다.
#
# 실제 Transformer에서는
#
#     Q = X @ W_Q
#     K = X @ W_K
#     V = X @ W_V
#
# 와 같이 학습된 행렬을 이용해 생성합니다.

Q = np.array([
    [1.0, 0.0],       # 철수
    [0.0, 1.0],       # 영희
    [1.0, 1.0],       # 책
    [0.8, 0.6]        # 주었다
])

K = np.array([
    [1.0, 0.0],       # 철수
    [0.0, 1.0],       # 영희
    [0.8, 0.8],       # 책
    [1.0, 0.5]        # 주었다
])

V = np.array([
    [1.0, 0.0],       # 철수의 정보
    [0.0, 1.0],       # 영희의 정보
    [1.0, 1.0],       # 책의 정보
    [0.5, 0.5]        # 주었다의 정보
])


# ============================================================
# 3. Attention 계산
# ============================================================

raw_scores, scores, weights, output = \
    scaled_dot_product_attention(Q, K, V)


# 숫자를 보기 쉽게 출력
np.set_printoptions(precision=3, suppress=True)


# ============================================================
# 4. 단계별 결과 출력
# ============================================================

print("=" * 60)
print("입력 토큰")
print("=" * 60)
print(tokens)


print("\n" + "=" * 60)
print("Q, K, V")
print("=" * 60)

print("\nQ =")
print(Q)

print("\nK =")
print(K)

print("\nV =")
print(V)


# ------------------------------------------------------------
# STEP 1
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 1. Query와 Key의 관련도 계산")
print("scores = Q @ K.T")
print("=" * 60)

print(raw_scores)

print("\nshape :", raw_scores.shape)


# ------------------------------------------------------------
# STEP 2
# ------------------------------------------------------------

d_k = Q.shape[-1]

print("\n" + "=" * 60)
print("STEP 2. Scaling")
print(f"scores / sqrt(d_k),  d_k = {d_k}")
print("=" * 60)

print(scores)


# ------------------------------------------------------------
# STEP 3
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 3. Softmax → Attention Weight")
print("=" * 60)

print(weights)

print("\n각 행의 합:")
print(weights.sum(axis=1))

print("\n※ 모든 행의 합이 1인지 확인합니다.")


# ------------------------------------------------------------
# STEP 4
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 4. Value의 가중합")
print("output = weights @ V")
print("=" * 60)

print(output)

print("\noutput shape :", output.shape)


# ============================================================
# 5. '주었다' 토큰 하나를 자세히 살펴보기
# ============================================================

i = 3       # "주었다"의 위치


print("\n\n" + "#" * 60)
print("        '주었다' 토큰의 계산을 자세히 살펴보기")
print("#" * 60)


# ------------------------------------------------------------
# ① Query 확인
# ------------------------------------------------------------

print("\n① '주었다'의 Query")
print(Q[i])


# ------------------------------------------------------------
# ② 각 Key와 dot product
# ------------------------------------------------------------

print("\n② 각 Key와의 Dot Product")

for j, token in enumerate(tokens):

    score = Q[i] @ K[j]

    print(
        f"주었다 ↔ {token:4s} : "
        f"{Q[i]} · {K[j]} = {score:.3f}"
    )


# ------------------------------------------------------------
# ③ Scaling 결과
# ------------------------------------------------------------

print("\n③ sqrt(d_k)로 나눈 결과")

for token, score in zip(tokens, scores[i]):

    print(
        f"주었다 ↔ {token:4s} : {score:.3f}"
    )


# ------------------------------------------------------------
# ④ Softmax 결과
# ------------------------------------------------------------

print("\n④ Softmax → Attention Weight")

for token, weight in zip(tokens, weights[i]):

    print(
        f"{token:4s} : {weight:.3f}"
    )

print(
    "\n가중치 합 =",
    weights[i].sum()
)


# ------------------------------------------------------------
# ⑤ Value에 가중치 적용
# ------------------------------------------------------------

print("\n⑤ 각 Value에 Attention Weight 적용")

weighted_values = []

for token, weight, value in zip(
        tokens, weights[i], V):

    weighted_value = weight * value

    weighted_values.append(weighted_value)

    print(
        f"{token:4s} : "
        f"{weight:.3f} × {value} "
        f"= {weighted_value}"
    )


# ------------------------------------------------------------
# ⑥ 모두 더하면 최종 출력
# ------------------------------------------------------------

print("\n⑥ 모두 더하면 '주었다'의 새로운 표현")

for value in weighted_values:
    print("+", value)

print("\n최종 output =", output[i])

### 실습2: 사전학습 Transformer로 문장 감정 분류하기

In [ ]:
# 처음 한 번만 설치
!pip install transformers torch

Tokenizer 내부 확인

In [ ]:
from transformers import AutoTokenizer


# 같은 모델의 tokenizer 불러오기
model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_id)


# 분석할 문장
text = "This class is very interesting."


# ============================================================
# 1. 문장을 token으로 분리
# ============================================================

tokens = tokenizer.tokenize(text)

print("원문:")
print(text)

print("\nTokens:")
print(tokens)


# ============================================================
# 2. token을 숫자 ID로 변환
# ============================================================

inputs = tokenizer(
    text,
    return_tensors="pt"
)

print("\nInput IDs:")
print(inputs["input_ids"])


# ============================================================
# 3. 전체 입력 구조 확인
# ============================================================

print("\n모델에 전달되는 입력:")
print(inputs)

In [ ]:
from transformers import pipeline


# ============================================================
# 1. 사전학습된 Transformer 모델 불러오기
# ============================================================

# pipeline()은
# 토큰화 → 모델 실행 → 결과 해석 과정을
# 간단하게 처리해 주는 Hugging Face 기능입니다.

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)


# ============================================================
# 2. Transformer에 입력할 문장 준비
# ============================================================

sentences = [
    "I really enjoyed this movie.",
    "This class is very interesting.",
    "The food was terrible."
]


# ============================================================
# 3. 문장을 Transformer 모델에 입력
# ============================================================

results = classifier(sentences)


# ============================================================
# 4. 결과 확인
# ============================================================

print("=" * 60)
print("Transformer 감정 분석 실습")
print("=" * 60)

for sentence, result in zip(sentences, results):

    print("\n문장 :", sentence)

    # POSITIVE : 긍정
    # NEGATIVE : 부정
    print("판정 :", result["label"])

    # 모델이 출력한 해당 label의 점수
    print("점수 :", round(result["score"], 4))


print("\n" + "=" * 60)
print("핵심")
print("=" * 60)

print("""
입력 문장
   ↓
Tokenizer
   ↓
Token IDs
   ↓
Transformer
   ↓
분류 Head
   ↓
POSITIVE / NEGATIVE
""")